# 02 · Exploratory Data Analysis (EDA) — SOL-USD
**Data Science Diploma · ENES UNAM León**

Objective: understand the distribution, trends, volatility, and temporal structure of the Solana price before building models.

> 💡 **Note for the technical report:** This notebook directly feeds section *6. Methodology* — here we justify preprocessing and feature engineering decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

# Load data
df = pd.read_csv('../data/raw/sol_usd_raw.csv', index_col=0, parse_dates=True)

# Flatten MultiIndex columns if generated by yfinance
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print(f'Data loaded: {df.shape} | {df.index.min().date()} → {df.index.max().date()}')
df.head(3)

## 2.1 Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Closing price histogram
axes[0].hist(df['Close'], bins=40, color='#9945FF', edgecolor='white', alpha=0.85)
axes[0].axvline(df['Close'].mean(), color='red', linestyle='--', label=f"Mean: ${df['Close'].mean():.1f}")
axes[0].axvline(df['Close'].median(), color='orange', linestyle='--', label=f"Median: ${df['Close'].median():.1f}")
axes[0].set_title('Closing Price Distribution')
axes[0].set_xlabel('USD')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['Close'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#9945FF', alpha=0.6))
axes[1].set_title('Closing Price Boxplot')
axes[1].set_ylabel('USD')

plt.tight_layout()
plt.savefig('../data/raw/02_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.2 Daily Returns and Volatility

ML models work better with **returns** than with absolute prices, since prices have a trend (they are not stationary).

In [ ]:
# Daily percentage return
df['daily_return'] = df['Close'].pct_change() * 100

# Rolling volatility (21-day rolling std ~ 1 month)
df['volatility_21d'] = df['daily_return'].rolling(21).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Daily returns
colors = ['#14F195' if r >= 0 else '#FF4444' for r in df['daily_return'].fillna(0)]
axes[0].bar(df.index, df['daily_return'], color=colors, alpha=0.8)
axes[0].axhline(0, color='white', linewidth=0.8)
axes[0].set_title('Daily Return (%) — SOL-USD')
axes[0].set_ylabel('%')

# Rolling volatility
axes[1].plot(df.index, df['volatility_21d'], color='#FFB800', linewidth=1.5)
axes[1].fill_between(df.index, df['volatility_21d'], alpha=0.2, color='#FFB800')
axes[1].set_title('21-Day Rolling Volatility (std dev of returns)')
axes[1].set_ylabel('Std Dev (%)')

plt.tight_layout()
plt.savefig('../data/raw/02_returns_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Average daily return : {df['daily_return'].mean():.3f}%")
print(f"Average volatility   : {df['daily_return'].std():.3f}%")
print(f"Worst day            : {df['daily_return'].min():.2f}%")
print(f"Best day             : {df['daily_return'].max():.2f}%")

## 2.3 OHLCV Variable Correlation

In [ ]:
corr_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
corr = df[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — OHLCV Variables')
plt.tight_layout()
plt.savefig('../data/raw/02_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.4 Stationarity Test (ADF Test)

Time series must be **stationary** for many models. We verify with the Augmented Dickey-Fuller test.

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    stationary = result[1] < 0.05
    print(f"\n{'='*50}")
    print(f"Series: {name}")
    print(f"  ADF Statistic : {result[0]:.4f}")
    print(f"  p-value       : {result[1]:.4f}")
    print(f"  Stationary?   : {'✅ YES' if stationary else '❌ NO — requires differencing'}")
    return stationary

adf_test(df['Close'], 'Close (absolute price)')
adf_test(df['daily_return'], 'Daily Return (%)')

## 2.5 Autocorrelation (ACF / PACF)

Shows how many past lags contain predictive information — this guides how many lags to use as features.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(df['Close'].dropna(), lags=30, ax=axes[0], color='#9945FF')
axes[0].set_title('ACF — Closing Price')

plot_pacf(df['Close'].dropna(), lags=30, ax=axes[1], color='#14F195', method='ywm')
axes[1].set_title('PACF — Closing Price')

plt.tight_layout()
plt.savefig('../data/raw/02_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print("💡 Interpretation:")
print("  - ACF: total correlation between price and its lags")
print("  - PACF: direct correlation (removing intermediate effects)")
print("  - Bars exceeding the blue band = significant lags → use as features")

## 2.6 Monthly Analysis

In [ ]:
df['month'] = df.index.month
df['month_name'] = df.index.strftime('%b %Y')

monthly_close = df.groupby(df.index.to_period('M'))['Close'].agg(['mean', 'min', 'max'])

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(monthly_close))
ax.fill_between(x, monthly_close['min'], monthly_close['max'], alpha=0.25, color='#9945FF', label='Min-Max Range')
ax.plot(x, monthly_close['mean'], color='#9945FF', linewidth=2, marker='o', markersize=5, label='Monthly Average')
ax.set_xticks(x)
ax.set_xticklabels([str(p) for p in monthly_close.index], rotation=45, ha='right')
ax.set_title('Monthly Price — SOL-USD (mean, min, max)')
ax.set_ylabel('USD')
ax.legend()
plt.tight_layout()
plt.savefig('../data/raw/02_monthly.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.7 Save Dataset with EDA Columns

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Save with return and volatility columns for use in the next notebook
df.drop(columns=['month', 'month_name'], inplace=True, errors='ignore')
df.to_csv('../data/processed/sol_usd_eda.csv')
print('Dataset with EDA features saved ✓')
print(df.columns.tolist())

---
## ✅ EDA Conclusions

| Finding | Implication for the model |
|---|---|
| Price is NOT stationary | Use returns or differences as features |
| Returns ARE stationary | Can be used directly |
| High autocorrelation at lags 1-5 | Include lags 1,2,3,5,7 as features |
| Variable volatility (clusters) | Add rolling volatility feature |
| High correlation between O/H/L/C | Normal in OHLCV, no problematic multicollinearity |

**Next step →** `03_feature_engineering.ipynb`